In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from morph import *

#3a, integrantes

mm.install()

img = cv2.imread('images/alunos.png')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

filtered = cv2.bilateralFilter(gray, 11, 75, 75)

ret, thresh = cv2.threshold(filtered, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

kernel = np.ones((5,5), np.uint8)
opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
sure_bg = cv2.dilate(opening, kernel, iterations=3)

dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)

ret, sure_fg = cv2.threshold(dist_transform, 0.2 * dist_transform.max(), 255, 0)
sure_fg = np.uint8(sure_fg)

unknown = cv2.subtract(sure_bg, sure_fg)
ret, markers = cv2.connectedComponents(sure_fg)

markers = markers + 1
markers[unknown == 255] = 0

markers_final = cv2.watershed(img, markers)

plt.figure(figsize=(15, 6))
plt.subplot(131), plt.imshow(opening, cmap='gray'), plt.title("Máscara Binária (3 silhuetas)")
plt.subplot(132), plt.imshow(dist_transform, cmap='magma'), plt.title("Mapa de Distância")
plt.subplot(133), mm.lblshow(markers_final), plt.title("Segmentação Final")
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from morph import *
from scipy import ndimage

mm.install()

#3a, avatares

img = cv2.imread('images/avatares.png')
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

s_channel = hsv[:,:,1]
v_channel = hsv[:,:,2]
_, mask_sat = cv2.threshold(s_channel, 25, 255, cv2.THRESH_BINARY)

gradient = cv2.morphologyEx(v_channel, cv2.MORPH_GRADIENT, np.ones((3,3), np.uint8))
_, mask_grad = cv2.threshold(gradient, 15, 255, cv2.THRESH_BINARY)

combined = cv2.bitwise_or(mask_sat, mask_grad)

kernel_join = cv2.getStructuringElement(cv2.MORPH_RECT, (5,5))
mask_closed = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kernel_join, iterations=3)

mask_filled = ndimage.binary_fill_holes(mask_closed).astype(np.uint8) * 255

dist_transform = cv2.distanceTransform(mask_filled, cv2.DIST_L2, 5)

ret, sure_fg = cv2.threshold(dist_transform, 0.45 * dist_transform.max(), 255, 0)
sure_fg = np.uint8(sure_fg)

sure_bg = cv2.dilate(mask_filled, np.ones((3,3), np.uint8), iterations=3)
unknown = cv2.subtract(sure_bg, sure_fg)

ret, markers = cv2.connectedComponents(sure_fg)
markers = markers + 1
markers[unknown == 255] = 0

markers_result = cv2.watershed(img, markers)

plt.figure(figsize=(15, 5))
plt.subplot(131), plt.imshow(mask_filled, cmap='gray'), plt.title("Máscara de Silhueta Íntegra")
plt.subplot(132), plt.imshow(dist_transform, cmap='magma'), plt.title("Mapa de Distância")
plt.subplot(133), mm.lblshow(markers_result), plt.title("Segmentação Final")
plt.show()